# **Computer Vision : Cat and Dog Classifiction** 🖼️

## 1️⃣ Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import ReduceLROnPlateau

## 2️⃣ Read Dataset

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    directory='/kaggle/input/datasets/salader/dogsvscats/train',
    validation_split=0.2,
    subset="training",
    seed=42,         
    batch_size=64,
    image_size=(128, 128)
)

val_ds = keras.utils.image_dataset_from_directory(
    directory='/kaggle/input/datasets/salader/dogsvscats/train',
    validation_split=0.2,
    subset="validation",
    seed=42,          
    batch_size=64,
    image_size=(128, 128)
)

test_ds = keras.utils.image_dataset_from_directory(
    directory='/kaggle/input/datasets/salader/dogsvscats/test',
    image_size=(128, 128),
    batch_size=64
)

In [ ]:
class_names = train_ds.class_names  # ['cats', 'dogs']
images, labels = next(iter(train_ds))

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample من كل Class', fontsize=16, fontweight='bold')

for class_idx, class_name in enumerate(class_names):
    class_images = images[labels == class_idx]
    
    for i in range(5):
        ax = axes[class_idx, i]
        img = class_images[i].numpy().astype('uint8')
        ax.imshow(img)
        ax.axis('off')
        if i == 0:
            ax.set_title(class_name, fontsize=13, fontweight='bold', loc='left')

plt.tight_layout()
plt.show()

## 3️⃣ Processing

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 4️⃣ Modeling

In [ ]:
model = Sequential([
    # Input layer to avoid warnings
    layers.Input(shape=(128, 128, 3)),
    layers.Rescaling(1./255),
    
    # Data Augmentation
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2), 
    layers.RandomZoom(0.2),    
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1), 
    layers.RandomContrast(0.3),

    # Block 1
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.35),

    # Block 3
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Block 
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Output Head
    layers.GlobalMaxPooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=5,
    min_lr=0.00001
)

model.compile(
    optimizer=AdamW(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    callbacks=[reduce_lr, early_stop],
    epochs=100,
    batch_size=64
)

In [ ]:
# Accuracy
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'])

plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

all_labels = []
all_preds = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    preds = (preds > 0.5).astype(int)

    all_labels.extend(labels.numpy())
    all_preds.extend(preds.flatten())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cat', 'Dog'],
            yticklabels=['Cat', 'Dog'])

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Sentinel V2: Confusion Matrix')
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['Cat', 'Dog']))

In [ ]:
model.save('/kaggle/working/model.keras')
print("saved")